# Federation Demo  
In this demo, you will be shown:  
- how to import data from endpoints
- how to see the datasets imported
- how to search for data
- merge data from different tables
- add a table to an existing database

In [ ]:
import csv
import os
from getpass import getpass
from pathlib import Path
from typing import Tuple, List, Dict, Any

import getpass
import paramiko
import shutil
import json
import pandas as pd
from pathlib import Path

from dsi.dsi import DSI
from dsi.dsifederated import DSIFederated
from dsi.sync import Sync
from dsi.utils.federated.federate_datasets import pull_data, just_pull_data

from dsi.utils.federation_utils import (
    compute_md5, 
    create_directory, 
    create_folder_from_path, 
    csv_to_list_of_dicts, 
    deduplicate_keep_latest, 
    get_last_part, 
    human_readable_size, 
    should_download, 
    upsert_records
)

Help for DSI and Federated DSI

In [ ]:
help(DSI)

In [ ]:
help(DSIFederated)

In [ ]:
help(Sync)

## Some Useful Functions

In [ ]:
def get_remote_endpoints(hostname, username, password, 
                         script_path='/users/pascalgrosset/dsi_test/load_dsi_endpoints.sh',
                         prefixes=['DSI_ENDPOINT_', 'DIANA_ENDPOINT_']):
    """Source bash script and get endpoints on remote server.
    
    Args:
        hostname: Remote server hostname
        username: SSH username
        password: SSH password
        script_path: Path to bash script on remote server
        prefixes: List of environment variable prefixes to match
    
    Returns:
        dict: Endpoint variables found on remote server
    """
    # Convert prefixes list to a format safe for bash
    prefixes_str = ','.join(f'"{p}"' for p in prefixes)
    
    # Use heredoc to avoid quote escaping issues
    command = f"""
source {script_path} && python3 << 'PYTHON_EOF'
import os
import json

# The prefixes we're looking for
prefixes = [{prefixes_str}]
prefix_tuple = tuple(prefixes)

# Get matching environment variables
endpoints = {{
    key: value 
    for key, value in os.environ.items() 
    if key.startswith(prefix_tuple)
}}

# Output as JSON so we can parse it easily
print(json.dumps(endpoints))
PYTHON_EOF
"""
    
    print(f"Connecting to {hostname}...")
    
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    
    try:
        ssh.connect(hostname, username=username, password=password, timeout=30)
        
        print(f"Sourcing {script_path} and reading endpoints...")
        stdin, stdout, stderr = ssh.exec_command(command)
        
        stdout_text = stdout.read().decode('utf-8').strip()
        stderr_text = stderr.read().decode('utf-8').strip()
        exit_code = stdout.channel.recv_exit_status()
        
        if exit_code != 0:
            print(f"✗ Command failed with exit code {exit_code}")
            if stderr_text:
                print(f"Error: {stderr_text}")
            return {}
        
        # Parse the JSON output
        endpoints = json.loads(stdout_text)
        
        print(f"✓ Found {len(endpoints)} endpoints:")
        for key, value in endpoints.items():
            print(f"  {key} = {value}")
        
        return endpoints
        
    except json.JSONDecodeError as e:
        print(f"✗ Failed to parse output: {e}")
        print(f"Raw output: {stdout_text}")
        return {}
    except Exception as e:
        print(f"!!!! !!!! !!! Error: {e}")
        return {}
    finally:
        try:
            ssh.close()
        except:
            pass

In [ ]:
def pull_remote_db(hpc_name:str, remote_dsi, db_folder) -> list:
    db_infos = []
    for key, value in remote_dsi.items():
        endpoint_name = key
        endpoint_db_path = value
        print(f"Retreiving data for {endpoint_name} at {endpoint_db_path}")
    
        username = input("Username: ")
        password = getpass.getpass("Password: ")  # Hidden input!
    
        db_info = pull_data(location_type="hpc",
                      location=hpc_name,
                      path=endpoint_db_path,
                      abs_path_workspace_folder=db_folder,
                      username=username,
                      password=password,
                      internal_use=False)
        db_infos.append(db_info)
    return db_infos

In [ ]:
def pull_remote_db(hpc_name:str, remote_dsi, temp_db_storage) -> list:
    db_infos = []
    for key, value in remote_dsi.items():
        endpoint_name = key
        endpoint_db_path = value
        print(f"Retreiving data for {endpoint_name} at {endpoint_db_path}")
    
        username = input("Username: ")
        password = getpass.getpass("Password: ")  # Hidden input!
    
        db_info = just_pull_data(location_type="hpc",
                      location=hpc_name,
                      path=endpoint_db_path,
                      abs_path_workspace_folder=temp_db_storage,
                      username=username,
                      password=password)
        db_infos.append(db_info)
    return db_infos
    

In [ ]:
def get_data_endpoints(default_endpoints_prefix=['DSI_ENDPOINT_', 'DIANA_ENDPOINT_']):
    prefix_tuple = tuple(default_endpoints_prefix)
    
    endpoints = {
        key: value 
        for key, value in os.environ.items() 
        if key.startswith(prefix_tuple)
    }

    print(endpoints)

    return endpoints     

In [ ]:
def read_dsi_sources_csv(filename: str) -> list:
    """Reads DSI sources CSV into a list of dictionaries."""
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            csv_reader = csv.DictReader(file)
            csv_data = list(csv_reader)
            print(f"✓ Loaded {len(csv_data)} records from {filename}")
            return csv_data
    except FileNotFoundError:
        print(f"✗ File not found: {filename}")
        return []
    except Exception as e:
        print(f"✗ Error reading {filename}: {e}")
        return []

In [ ]:
def combine_datasources(endpoints):
    dsi_sources = []
    for key, value in endpoints.items():
        print(f"Key: {key}, Value: {value}")
        sources = read_dsi_sources_csv(value)
        dsi_sources.extend(sources)

    return dsi_sources

In [ ]:
def read_data_sources(csv_data, workspace_folder) -> Tuple[List[Dict[str, Any]], int]:
    database_info = []
    federation_dbs = []
    success_counter = 0
    for row in csv_data:
        username = ""
        password = ""
        if row['location_type'].strip().lower() == "hpc":
            print(f"\n{'='*60}")
            print(f"Enter credentials for data at {row['location']} : {row['path']}")
            username = input("Username: ")
            password = getpass.getpass("Password: ")  # Hidden input!
    
        db_info = pull_data(location_type=row['location_type'],
                  location=row['location'],
                  path=row['path'],
                  abs_path_workspace_folder=workspace_folder,
                  username=username,
                  password=password,
                  internal_use=False)
        
        if db_info:
            database_info.append(db_info)
            combined = {k: row[k] for k in ["location_type", "location", "submitter_name"]} | {k: db_info[k] for k in ["local_path", "name", "folder_hash"]}
            combined["workspace_folder"] = workspace_folder
            federation_dbs.append(combined)
            success_counter += 1

    # Save databases information to a JSON file
    upsert_records(f"{workspace_folder}/dsi_database_list.json", database_info, key="original_path")

    return database_info, success_counter

In [ ]:
def create_folder(folder_name:str):
    folder_path = Path(folder_name)
    
    # Delete if exists
    if folder_path.exists():
        shutil.rmtree(folder_path)
    
    # Create the folder
    folder_path.mkdir(parents=True, exist_ok=True)

In [ ]:
def combine_csv(folder_path:str, output_csv) -> list:
    csv_files = Path(folder_path).glob("*.csv")
    
    dfs = []
    for file in csv_files:
        df = pd.read_csv(file)
        df['source_file'] = file.name  # Add column with source filename
        dfs.append(df)
    
    combined_df = pd.concat(dfs, ignore_index=True)
    combined_df.to_csv(output_csv, index=False)

    return combined_df.to_dict('records')

## Get Enpoints from Chicoma

In [ ]:
username = input("Enter username: ")
password = getpass.getpass("Enter password: ")
endpoints_location = get_remote_endpoints("ch-fe.lanl.gov", username, password)

In [ ]:
endpoints_location

### Pull the databases

In [ ]:
temp_db_storage = "test_00"
create_folder(temp_db_storage)

pull_remote_db("ch-fe.lanl.gov", endpoints_location, temp_db_storage)

In [ ]:
output_csv = "output_csv.csv"
csv_data_sources = combine_csv(temp_db_storage, output_csv)

### Federate the data in specified folder

In [ ]:
rel_wrks_folder = "test_federate_03"
workspace_folder = str(Path(rel_wrks_folder).resolve())

In [ ]:
database_info = read_data_sources(csv_data_sources, workspace_folder)

In [ ]:
database_info

## Instantiate the object

In [ ]:
federated_dbs = DSIFederated(workspace_folder, operating_mode="notebook")

## Browse and search through the data

In [ ]:
federated_dbs.f_list_databases()

## Looking up data

In [ ]:
federated_dbs.f_summary()

In [ ]:
federated_dbs.f_search(query="dens_max")

## Merging data

### Search for databases

In [ ]:
federated_dbs.f_search_for_databases(db="data_subset*")

### Search for the path to a database

In [ ]:
federated_dbs.f_get_db_path(db="data_subset_1.db")

### Load that database

In [ ]:
temp_file = DSI('/Users/pascalgrosset/projects/dsi/dsi_databases_00/87c1361f3c2316dd/data_subset_1.db')

In [ ]:
temp_file.list()

In [ ]:
temp_file.get_table("data", collection=True)

In [ ]:
federated_dbs.f_merge(src_db_id='camouflaged-hare',src_tbl_name='data', dst_db_id='aromatic-dragon',dst_tbl_name='data')

In [ ]:
temp_file.get_table("data", collection=True)

In [ ]:
federated_dbs.f_merge(src_db_id='precious-walrus',src_tbl_name='data', dst_db_id='aromatic-dragon',dst_tbl_name='data')

In [ ]:
temp_file.get_table("data", collection=True)

## Adding another table to the database

In [ ]:
federated_dbs.f_search_for_databases(db="model_subset*")

In [ ]:
federated_dbs.f_add_table(src_db_id='satisfied-whale',src_tbl_name='data', dst_db_id='aromatic-dragon',dst_tbl_name='model')

In [ ]:
temp_file.list()

In [ ]:
temp_file.get_table("model", collection=True)

That database now has datasets which have been pulled from different sites as well as several tables 